# Notebook 10 — Revision-01 robustness checks (reviewer points 3.4, 3.5, 3.7)

* **(3) Sec. 3.5** — sensitivity of $(a, b)$ in $M_1^{\star}$ to the filter $\rho \in [\rho_{\min}, \rho_{\max}]$.
* **(4) Sec. 3.7** — sensitivity to the weighting scheme $w$.
* **(5) Sec. 3.4** — block-bootstrap over $N$ (resampling whole $N$-histograms) vs. the original point bootstrap.

Uses the two-parameter primary model $M_1^{\star}$ selected in notebook 09. Writes `paper-rev-01/rev01_robustness.json`.

In [1]:
import math, json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import integrate

WORK = Path.cwd()
DATA_DIR = WORK / 'ml_data'
OUT_DIR = WORK / 'paper-rev-01'; OUT_DIR.mkdir(exist_ok=True)
C2_TWIN = 0.6601618158468695739278121100145557784326233602847334133194484233354

def hl_correct(g):
    if g <= 0 or g % 2: return 0.0
    n = g
    while n % 2 == 0: n //= 2
    corr, p = 1.0, 3
    while p*p <= n:
        if n % p == 0:
            corr *= (p-1)/(p-2)
            while n % p == 0: n //= p
        p += 2
    if n > 1: corr *= (n-1)/(n-2)
    return 2*C2_TWIN*corr

def omega(n):
    if n <= 1: return 0
    c, d = 0, 2
    while d*d <= n:
        if n % d == 0:
            c += 1
            while n % d == 0: n //= d
        d += 1
    if n > 1: c += 1
    return c

def li2(N):
    v, _ = integrate.quad(lambda t: 1.0/np.log(t)**2, 2.0, N, limit=200)
    return float(v)

cached = sorted(DATA_DIR.glob('gaps_N*.csv'))
histograms = {int(f.stem.replace('gaps_N','')): pd.read_csv(f) for f in cached}
print(f'{len(histograms)} histograms, N in [{min(histograms):.0e}, {max(histograms):.0e}]')

25 histograms, N in [1e+05, 1e+13]


In [2]:
def build(rho_min=0.05, rho_max=1.10, weight_scheme='min1_emp1000'):
    rows = []
    for N, df in histograms.items():
        lnN = math.log(N); lnlnN = math.log(lnN); li2_N = li2(N)
        for _, r in df.iterrows():
            g, emp = int(r['gap']), int(r['count'])
            if g < 2 or g > 100 or g % 2 or emp < 100: continue
            Cg = hl_correct(g); rho = g*Cg/lnN
            if not (rho_min <= rho <= rho_max): continue
            if   weight_scheme == 'min1_emp1000':   w = min(1.0, emp/1000.0)
            elif weight_scheme == 'min1_emp100':    w = min(1.0, emp/100.0)
            elif weight_scheme == 'min1_emp5000':   w = min(1.0, emp/5000.0)
            elif weight_scheme == 'sqrt_emp':       w = math.sqrt(emp)
            elif weight_scheme == 'uniform':        w = 1.0
            elif weight_scheme == 'emp':            w = float(emp)
            else: raise ValueError(weight_scheme)
            rows.append({
                'N': N, 'g': g,
                'log_empir': math.log(emp),
                'log_C_li': math.log(Cg*li2_N),
                'rho': rho,
                'sqrt_omega_g': math.sqrt(omega(g)),
                'lnlnN': lnlnN,
                'weight': w,
            })
    return pd.DataFrame(rows)

def fit_M1star(D):
    """Two-parameter model: R = a*sqrt(omega) + b*lnlnN, no intercept."""
    y = D['log_empir'].values
    offset = D['log_C_li'].values - D['rho'].values
    target = y - offset
    X = np.column_stack([D['sqrt_omega_g'].values, D['lnlnN'].values])
    w = D['weight'].values
    ws = np.sqrt(w)
    coefs, *_ = np.linalg.lstsq(X*ws[:,None], target*ws, rcond=None)
    return coefs

## (3) Sensitivity to the filter $\rho \in [\rho_{\min}, \rho_{\max}]$

In [3]:
filters = [
    (0.05, 1.10),
    (0.10, 1.00),
    (0.20, 1.00),
    (0.33, 0.95),
    (0.05, 0.95),
    (0.10, 1.10),
]
filter_results = []
print(f"{'rho_min':>8s} {'rho_max':>8s} {'n':>4s} {'a':>9s} {'b':>9s}")
for rm, rM in filters:
    D = build(rm, rM)
    a, b = fit_M1star(D)
    filter_results.append({'rho_min': rm, 'rho_max': rM, 'n': len(D),
                           'a': float(a), 'b': float(b)})
    print(f'{rm:8.2f} {rM:8.2f} {len(D):4d} {a:+9.4f} {b:+9.4f}')

 rho_min  rho_max    n         a         b
    0.05     1.10  140   +0.8552   -0.2017
    0.10     1.00  124   +0.8032   -0.1840


    0.20     1.00  102   +0.7264   -0.1476


    0.33     0.95   79   +0.6882   -0.1311
    0.05     0.95  122   +0.8082   -0.1879


    0.10     1.10  136   +0.8371   -0.1939


## (4) Sensitivity to the weighting scheme

In [4]:
schemes = ['min1_emp100', 'min1_emp1000', 'min1_emp5000', 'sqrt_emp', 'uniform', 'emp']
weight_results = []
print(f"{'scheme':>18s} {'n':>4s} {'a':>9s} {'b':>9s}")
for s in schemes:
    D = build(0.05, 1.10, weight_scheme=s)
    a, b = fit_M1star(D)
    weight_results.append({'scheme': s, 'n': len(D), 'a': float(a), 'b': float(b)})
    print(f'{s:>18s} {len(D):4d} {a:+9.4f} {b:+9.4f}')

            scheme    n         a         b


       min1_emp100  140   +0.8554   -0.2018
      min1_emp1000  140   +0.8552   -0.2017


      min1_emp5000  140   +0.8554   -0.2018


          sqrt_emp  140   +0.7307   -0.1603
           uniform  140   +0.8554   -0.2018


               emp  140   +0.7835   -0.1774


## (5) Block-bootstrap over $N$

Resample whole $N$-histograms with replacement (instead of individual points). 23 blocks of size $\approx 5$ each; $B = 1000$.

In [5]:
D = build(0.05, 1.10)
N_values = sorted(D['N'].unique())
blocks = {N: D[D['N']==N].reset_index(drop=True) for N in N_values}
print(f'{len(blocks)} N-blocks; block sizes: {[len(v) for v in blocks.values()]}')

rng = np.random.default_rng(42)
B = 1000
coefs_block = np.empty((B, 2))
point_block = fit_M1star(D)
for b in range(B):
    pick = rng.choice(N_values, size=len(N_values), replace=True)
    Db = pd.concat([blocks[N] for N in pick], ignore_index=True)
    coefs_block[b] = fit_M1star(Db)

# Point-bootstrap for comparison.
coefs_point = np.empty((B, 2))
n = len(D)
for b in range(B):
    idx = rng.integers(0, n, size=n)
    coefs_point[b] = fit_M1star(D.iloc[idx])

def pct(x): return tuple(float(v) for v in np.percentile(x, [5, 50, 95]))

print(f"\n{'method':>18s} {'param':>6s} {'point':>9s} {'5%':>9s} {'50%':>9s} {'95%':>9s} {'width':>9s}")
for method, coefs in [('point-bootstrap', coefs_point), ('block-bootstrap (N)', coefs_block)]:
    for i, lab in enumerate(['a', 'b']):
        lo, med, hi = pct(coefs[:, i])
        print(f'{method:>18s} {lab:>6s} {point_block[i]:+9.4f} '
              f'{lo:+9.4f} {med:+9.4f} {hi:+9.4f} {hi-lo:9.4f}')

25 N-blocks; block sizes: [3, 3, 3, 3, 3, 4, 4, 5, 5, 5, 5, 5, 6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 9]



            method  param     point        5%       50%       95%     width
   point-bootstrap      a   +0.8552   +0.7905   +0.8552   +0.9213    0.1308
   point-bootstrap      b   -0.2017   -0.2270   -0.2016   -0.1772    0.0498
block-bootstrap (N)      a   +0.8552   +0.7906   +0.8546   +0.9334    0.1428
block-bootstrap (N)      b   -0.2017   -0.2301   -0.2016   -0.1790    0.0510


In [6]:
out = {
    'filter_scan': filter_results,
    'weight_scan': weight_results,
    'bootstrap_comparison': {
        'B': B,
        'point_estimate': {'a': float(point_block[0]), 'b': float(point_block[1])},
        'point_bootstrap': {
            'a_5_50_95': list(pct(coefs_point[:,0])),
            'b_5_50_95': list(pct(coefs_point[:,1])),
        },
        'block_bootstrap_N': {
            'a_5_50_95': list(pct(coefs_block[:,0])),
            'b_5_50_95': list(pct(coefs_block[:,1])),
        },
    },
}
(OUT_DIR / 'rev01_robustness.json').write_text(json.dumps(out, indent=2))
print('Saved:', OUT_DIR / 'rev01_robustness.json')

Saved: /opt/apps/jupyter/work/paper-rev-01/rev01_robustness.json
